# Capítulo 13: Regressão Logística

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto. É o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem daqui —
# no livro isso vem do `execute-dir: project` do Quarto.
import os
import sys

_raiz = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_raiz, "_quarto.yml")):
    _pai = os.path.dirname(_raiz)
    if _pai == _raiz:
        raise RuntimeError("raiz do projeto não encontrada (procurando _quarto.yml)")
    _raiz = _pai
os.chdir(_raiz)
if _raiz not in sys.path:
    sys.path.insert(0, _raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **📌 Nota**
>
> Este capítulo corresponde ao capítulo 16 de Grus (2019).

> Muita gente diz que existe uma linha tênue entre a genialidade e a loucura. Eu não acho que a linha seja tênue — acho que existe um abismo entre as duas.
>
> — Bill Bailey

O [Capítulo 12](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/index.html) terminou com um modelo que prevê **quanto**: minutos por dia, um número contínuo, com um coeficiente por variável explicativa. Este capítulo troca a pergunta por outra, que na prática é a mais comum das duas: **sim ou não**. O usuário pagou pela conta premium? A mensagem é spam? O candidato passa na entrevista? A variável a prever deixa de ser uma quantidade e passa a ser um rótulo, codificado como 0 ou 1.

A [seção 13.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/01-o-problema.html) faz a coisa óbvia: pega o modelo linear do capítulo anterior, sem alterar uma linha, e o aponta para um alvo 0/1. Ele roda, devolve coeficientes e prevê — números negativos, e números maiores que 1, para uma variável que só assume os valores 0 e 1. É um fracasso instrutivo, e ele define o problema que o resto do capítulo resolve: a saída precisa ficar presa no intervalo $[0, 1]$ para poder ser lida como probabilidade.

Este capítulo também cobra duas dívidas antigas. A primeira é da [seção 5.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/05-ajustando-modelos.html), que prometeu que a regressão logística **não tem fórmula fechada** — não existem duas médias e uma covariância que resolvam este problema, e nem sequer um sistema linear a resolver, como havia na regressão múltipla. O gradiente descendente deixa de ser o caminho mais prático entre dois possíveis e passa a ser o **único** caminho: sem ele, não há modelo. A segunda dívida é da [seção 11.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/03-maxima-verossimilhanca.html), que introduziu estimação por máxima verossimilhança e mostrou que, para erros normais, maximizar a verossimilhança é exatamente minimizar a soma dos quadrados. Aqui os erros não são normais — o alvo é binário —, aquela equivalência some, e sobra a verossimilhança sozinha. É ela que vamos otimizar, na forma em que um computador consegue: **minimizando a log-verossimilhança negativa**.

E há um terceiro fio, que só se enxerga na última seção. Ajustar $\beta$ produz, de brinde, um **hiperplano** que separa as duas classes — o conjunto de pontos onde `dot(x, beta)` vale zero. Encontrar diretamente o hiperplano que melhor separa as classes, sem passar por probabilidade nenhuma, é um critério diferente e dá origem a outra família de classificadores: as **máquinas de vetores de suporte**. A [seção 13.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/05-maquinas-de-vetores-de-suporte.html) fecha o capítulo comparando os dois critérios.

Ao final deste capítulo, você será capaz de:

- Explicar por que um modelo linear não serve para prever uma variável binária, e demonstrar isso com previsões fora do intervalo $[0, 1]$
- Escrever a função logística, justificar por que ela resolve o problema do alcance e derivar a sua derivada
- Derivar a log-verossimilhança de um modelo com alvo binário e explicar por que maximizá-la equivale a minimizar a log-verossimilhança negativa
- Ajustar uma regressão logística por gradiente descendente, sem nenhuma fórmula fechada disponível
- Explicar por que um ajuste pode reportar perda zero e estar completamente errado, e por que nenhuma biblioteca teria mostrado isso a você
- Avaliar um classificador probabilístico com precisão e revocação, e explicar de onde vem o limiar de 0,5
- Descrever o critério de margem máxima de uma máquina de vetores de suporte e dizer em que ele difere de maximizar a verossimilhança

## Seções

| Seção | Tópico |
|---|---|
| [13.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/01-o-problema.html) | O Problema |
| [13.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/02-a-funcao-logistica.html) | A Função Logística |
| [13.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/03-aplicando-o-modelo.html) | Aplicando o Modelo |
| [13.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/04-qualidade-do-ajuste.html) | Qualidade do Ajuste |
| [13.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/05-maquinas-de-vetores-de-suporte.html) | Máquinas de Vetores de Suporte |

## O Problema

> **📌 Nota**
>
> Esta seção corresponde a *The Problem*, do capítulo 16 de Grus (2019).

O [Capítulo 1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap01/03-hipotese-motivadora-datasciencester.html) já encostou neste problema. Lá, com um punhado de usuários e nenhuma técnica, olhamos anos de experiência contra conta paga, vimos que os extremos pagavam e o meio não, e escrevemos um classificador com os cortes chutados à mão — `if anos < 3.0 ... elif anos < 8.5 ...`. Aquele código foi apresentado como o que era: um modelo que lê os cortes dos próprios dados que deveria explicar, e que não sobrevive ao primeiro usuário novo.

Agora temos ferramentas. Vamos refazer o problema direito.

### Os dados

O conjunto tem cerca de 200 usuários anonimizados, e para cada um sabemos três coisas: **anos de experiência** como cientista de dados, **salário**, e se a pessoa **pagou** pela conta premium. Como é típico com variáveis categóricas, o alvo é representado como 0 (não pagou) ou 1 (pagou) — a mesma variável indicadora que a [seção 12.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/01-o-modelo.html) usou para "tem doutorado", agora do lado de fora do modelo, como a coisa a prever.

Cada linha bruta é `[experiência, salário, conta_paga]`. Convertendo para o formato de que precisamos — com a coluna de 1 na frente, pela convenção do capítulo anterior:

```python
xs = [[1.0] + row[:2] for row in data]    # [1, experiência, salário]
ys = [row[2] for row in data]             # conta_paga
```

In [ ]:
import random
random.seed(0)
from scratch.logistic_regression import xs, ys
from scratch.multiple_regression import least_squares_fit, predict
import matplotlib.pyplot as plt
plt.close('all')

In [ ]:
len(xs), xs[0], ys[0], sum(ys)

São 200 usuários, dos quais 52 pagaram — 26% da base. A primeira linha é um usuário com 0,7 ano de experiência, salário de 48.000 e conta paga.

> **📌 Nota — Por que o `import` está escondido**
>
> O chunk que importa está marcado com `#| include: false`, pelo mesmo motivo dos capítulos 11 e 12: `scratch/multiple_regression.py` importa `scratch/statistics.py` e `scratch/probability.py`, e esses dois têm chamadas `plt.*` soltas no nível do módulo — cinco e dezoito, respectivamente. Sem o `plt.close('all')` logo depois, figuras que ninguém pediu apareceriam no meio da página. O `random.seed(0)` antes do `import` existe porque `multiple_regression` também roda, no nível do módulo, um trecho de reamostragem aleatória sem semente.
>
> `scratch/logistic_regression.py`, por outro lado, é limpo: ele define os dados e as funções deste capítulo e não desenha nada na importação.

Vale olhar os dados antes de modelar:

In [ ]:
# Figura: Usuários pagantes e não pagantes
experiencia = [x[1] for x in xs]
salario = [x[2] for x in xs]

plt.scatter([e for e, y in zip(experiencia, ys) if y == 1],
            [s for s, y in zip(salario, ys) if y == 1],
            marker='+', label='paga')
plt.scatter([e for e, y in zip(experiencia, ys) if y == 0],
            [s for s, y in zip(salario, ys) if y == 0],
            marker='.', label='não paga')
plt.xlabel("anos de experiência")
plt.ylabel("salário")
plt.legend(loc='upper left')
plt.title("Usuários pagantes e não pagantes")
plt.show()

Os pagantes ficam quase todos **abaixo** da nuvem principal, e sobretudo à direita dela: **muita experiência e salário relativamente baixo**. Faz algum sentido — quem tem experiência e ainda não está ganhando bem talvez esteja procurando emprego, que é para o que serve uma conta premium numa rede de cientistas de dados. Há também um punhado de pagantes no canto inferior esquerdo, com pouca experiência e os salários mais baixos do conjunto.

Guarde essa observação; ela vai reaparecer como um coeficiente negativo na [seção 13.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/03-aplicando-o-modelo.html), e o sinal negativo assusta quem não olhou o gráfico antes.

### A tentativa óbvia

A primeira tentativa óbvia é usar regressão linear e achar o melhor modelo:

$$
\text{conta paga} = \beta_0 + \beta_1 \,\text{experiência} + \beta_2 \,\text{salário} + \varepsilon
$$

Nada impede de modelar o problema assim. Literalmente nada: a função `least_squares_fit` do [Capítulo 12](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/03-ajustando-o-modelo.html) aceita qualquer lista de `ys`, e não tem como saber que estes só valem 0 e 1.

Antes de ajustar, um passo de higiene: **reescalonar**. Experiência vai de 0,1 a 10; salário vai de 30.000 a 107.000. Com amplitudes que diferem por um fator de quase 8.000, a coluna do salário domina qualquer produto escalar, e o gradiente descendente com taxa de aprendizado única não tem chance. A função é a mesma que a [seção 7.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/06-reescalonamento.html) construiu; como aquele módulo não pode ser importado neste livro, escrevemos as duas funções aqui:

In [ ]:
from typing import List, Tuple
from scratch.linear_algebra import Vector, vector_mean
from scratch.statistics import standard_deviation

def scale(data: List[Vector]) -> Tuple[Vector, Vector]:
    """devolve a média e o desvio padrão de cada posição"""
    dim = len(data[0])
    means = vector_mean(data)
    stdevs = [standard_deviation([vector[i] for vector in data])
              for i in range(dim)]
    return means, stdevs

def rescale(data: List[Vector]) -> List[Vector]:
    """cada posição passa a ter média 0 e desvio padrão 1"""
    dim = len(data[0])
    means, stdevs = scale(data)
    rescaled = [v[:] for v in data]
    for v in rescaled:
        for i in range(dim):
            if stdevs[i] > 0:
                v[i] = (v[i] - means[i]) / stdevs[i]
    return rescaled

rescaled_xs = rescale(xs)
[round(v, 4) for v in rescaled_xs[0]]

A coluna de 1 sobrevive intacta: o desvio padrão dela é zero, e `rescale` deixa em paz qualquer dimensão sem variação em vez de dividir por zero.

> **❗ Importante — Reescalonar aqui é higiene; na próxima seção vira sobrevivência**
>
> Neste ajuste linear, esquecer o `rescale` produziria um modelo ruim — coeficientes que não convergem, uma taxa de aprendizado que serve para uma coluna e não para a outra. Chato, mas visível: o número sai errado e você percebe.
>
> Na [seção 13.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/03-aplicando-o-modelo.html), com o mesmo conjunto de dados e o mesmo tipo de otimizador, esquecer este passo não produz um modelo ruim. Produz um `ValueError` na primeira conta, antes do primeiro passo de gradiente. Vale voltar a esta linha depois de ver aquilo acontecer.

Agora o ajuste, com o `least_squares_fit` do capítulo anterior, sem uma linha de mudança — e com o `#| warning: false` que a [seção 12.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/03-ajustando-o-modelo.html) explica, porque `least_squares_fit` usa `tqdm` por dentro:

In [ ]:
from scratch.multiple_regression import least_squares_fit, predict

random.seed(0)
learning_rate = 0.001
beta = least_squares_fit(rescaled_xs, ys, learning_rate, 1000, 1)

[round(b, 4) for b in beta]

O modelo rodou e devolveu coeficientes. Vamos ver as previsões contra os valores reais:

In [ ]:
# Figura: Regressão linear para prever contas pagas
previsoes = [predict(x_i, beta) for x_i in rescaled_xs]

plt.scatter(previsoes, ys, marker='.')
plt.axvline(0, color='gray', linestyle=':')
plt.axvline(1, color='gray', linestyle=':')
plt.xlabel("previsto")
plt.ylabel("real")
plt.title("Regressão linear para prever contas pagas")
plt.show()

### Os dois problemas

O primeiro salta do gráfico. As duas linhas pontilhadas marcam 0 e 1, os únicos valores que a variável a prever assume. Trinta e oito dos 200 pontos caem fora desse intervalo:

In [ ]:
negativas = sum(1 for p in previsoes if p < 0)
acima_de_um = sum(1 for p in previsoes if p > 1)

print(f"previsões negativas:  {negativas}")
print(f"previsões acima de 1: {acima_de_um}")
print(f"mínimo: {min(previsoes):.4f}   máximo: {max(previsoes):.4f}")

Gostaríamos que a saída ficasse entre 0 e 1, para poder lê-la como probabilidade — uma previsão de 0,25 significaria "25% de chance de ser assinante". Isso seria perfeitamente aceitável. Mas as previsões deste modelo vão de **−0,58** a **1,44**: 37 dos 200 usuários recebem uma previsão negativa, e um recebe previsão acima de 1. Não há leitura possível para uma probabilidade negativa.

O segundo problema é mais sutil, e é sobre a validade do ajuste, não sobre a interpretação da saída. A regressão linear supõe que os erros são **não correlacionados** com as colunas de `x` — é uma das hipóteses que a [seção 12.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/02-hipoteses-do-minimos-quadrados.html) listou. Aqui essa hipótese é violada por construção. O coeficiente de experiência é positivo (mais experiência, maior previsão), então o modelo produz valores altos para quem tem muita experiência. Só que o valor real é, no máximo, 1. Logo, uma previsão muito alta **obriga** o erro a ser muito negativo — o erro passa a depender de `x`, exatamente o que a hipótese proibia. A consequência não é cosmética: a estimativa de $\beta$ fica **enviesada**.

> **🔷 Conceito**
>
> O que queremos, no lugar disso: que valores grandes e positivos de `dot(x_i, beta)` correspondam a probabilidades **próximas de 1**, e valores grandes e negativos, a probabilidades **próximas de 0** — com uma transição suave no meio.
>
> Repare no que essa formulação **não** pede. Ela não pede um modelo diferente, nem um otimizador diferente, nem abandonar o produto escalar. `dot(x_i, beta)` continua sendo o coração da coisa; a estrutura linear do capítulo anterior sobrevive inteira. O que falta é uma função aplicada **depois** dele, que pegue a reta inteira dos reais e a comprima dentro de $[0, 1]$. É essa função que a [próxima seção](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/02-a-funcao-logistica.html) apresenta.

> **💡 Dica — Na prática: `scikit-learn`**
>
> O ajuste que acabamos de fazer tem nome na literatura estatística — **modelo de probabilidade linear** — e não é uma bobagem completa: quando as probabilidades envolvidas ficam longe de 0 e de 1, ele aproxima razoavelmente bem e tem a vantagem de coeficientes diretamente interpretáveis. Fora dessa faixa, ele é o que você viu no gráfico.
>
> O ponto para esta seção é outro: **a biblioteca não avisa**. `LinearRegression().fit(X, y)` com `y` binário roda, converge, devolve `coef_` e `score`, e nenhuma linha de aviso menciona que o alvo só tem dois valores:
>
> ```python
> from sklearn.linear_model import LinearRegression
>
> X = [x[1:] for x in xs]      # sem a coluna de 1
> modelo = LinearRegression().fit(X, ys)
> modelo.predict(X).min()      # negativo
> ```
>
> A escolha certa para um alvo binário é `LogisticRegression` — que é o que este capítulo constrói do zero, e que aparece no callout da [seção 13.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/03-aplicando-o-modelo.html). Escolher entre as duas é decisão de quem modela, não da biblioteca.

## A Função Logística

> **📌 Nota**
>
> Esta seção corresponde a *The Logistic Function*, do capítulo 16 de Grus (2019).

A seção anterior terminou com um pedido preciso: uma função que receba a reta inteira dos reais e devolva um número em $[0, 1]$, crescente e suave. A escolha da regressão logística é a **função logística**:

In [ ]:
import math

def logistic(x: float) -> float:
    return 1.0 / (1 + math.exp(-x))

Uma linha. É útil ver o formato dela antes de qualquer conta:

In [ ]:
# Figura: A função logística
from matplotlib import pyplot as plt

grade = [x / 10 for x in range(-100, 101)]

plt.plot(grade, [logistic(x) for x in grade])
plt.axhline(0, color='gray', linewidth=0.8)
plt.axhline(1, color='gray', linewidth=0.8)
plt.axhline(0.5, color='gray', linestyle=':', linewidth=0.8)
plt.axvline(0, color='gray', linestyle=':', linewidth=0.8)
plt.xlabel("x")
plt.ylabel("logistic(x)")
plt.title("A função logística")
plt.show()

Conforme a entrada fica grande e positiva, a saída chega cada vez mais perto de 1; conforme fica grande e negativa, chega cada vez mais perto de 0. Em zero ela vale exatamente 0,5. É exatamente a forma pedida na seção anterior.

Ela tem também uma propriedade conveniente: a derivada se escreve em função da própria função.

In [ ]:
def logistic_prime(x: float) -> float:
    y = logistic(x)
    return y * (1 - y)

Isso vai importar daqui a pouco — é o que torna o gradiente deste modelo simples o bastante para caber em uma linha.

Com ela, o modelo fica:

$$
y_i = f(\mathbf{x}_i \cdot \beta) + \varepsilon_i
$$

onde $f$ é a função logística. Repare no que **não** mudou: $\mathbf{x}_i \cdot \beta$ continua ali, com a coluna de 1 e tudo. A estrutura linear do [Capítulo 12](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/index.html) sobreviveu inteira; ganhou uma função por cima.

### Por que minimizar quadrados deixa de ser a mesma coisa

Na regressão linear, ajustamos o modelo minimizando a soma dos erros ao quadrado, e a [seção 11.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/03-maxima-verossimilhanca.html) mostrou que isso acabava escolhendo o $\beta$ que **maximiza a verossimilhança dos dados** — não por coincidência, mas como consequência de supor erros normais.

Aqui as duas coisas deixam de ser equivalentes. O alvo é binário: um usuário paga ou não paga, e o "erro" de um ponto não é um desvio normalmente distribuído em torno de uma reta — é a diferença entre um rótulo em $\{0, 1\}$ e uma probabilidade em $[0, 1]$. Supor erros normais aqui seria supor algo que os dados desmentem por construção. Some a suposição, some a equivalência, e sobra apenas um dos dois lados: **a verossimilhança**. Vamos maximizá-la diretamente, por gradiente descendente.

Dado um $\beta$, o modelo diz que cada $y_i$ vale 1 com probabilidade $f(\mathbf{x}_i \cdot \beta)$ e 0 com probabilidade $1 - f(\mathbf{x}_i \cdot \beta)$. Essas duas frases se juntam numa só expressão:

$$
p(y_i \mid \mathbf{x}_i, \beta) = f(\mathbf{x}_i \cdot \beta)^{y_i} \, \bigl(1 - f(\mathbf{x}_i \cdot \beta)\bigr)^{1 - y_i}
$$

O truque é o expoente. Se $y_i$ é 0, o primeiro fator vira $f^0 = 1$ e sobra $1 - f(\mathbf{x}_i \cdot \beta)$; se $y_i$ é 1, o segundo fator vira 1 e sobra $f(\mathbf{x}_i \cdot \beta)$. Uma fórmula, os dois casos, sem `if`.

Essa expressão tem nome: é a distribuição de **Bernoulli** de parâmetro $f(\mathbf{x}_i \cdot \beta)$ — a distribuição de uma única moeda viciada. É exatamente a troca que a [seção 11.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/03-maxima-verossimilhanca.html) anunciou ao dizer que a máxima verossimilhança sobreviveria à mudança de alvo, mas a distribuição suposta para os dados não: lá era normal, aqui é Bernoulli. O método é o mesmo; a densidade que entra nele é outra, e é dela que vem tudo o que muda daqui para a frente.

Como na [seção 11.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/03-maxima-verossimilhanca.html), é mais simples trabalhar com o **logaritmo** da verossimilhança:

$$
\log L(\beta \mid \mathbf{x}_i, y_i) = y_i \log f(\mathbf{x}_i \cdot \beta) + (1 - y_i) \log \bigl(1 - f(\mathbf{x}_i \cdot \beta)\bigr)
$$

Como o log é estritamente crescente, qualquer $\beta$ que maximize a log-verossimilhança maximiza também a verossimilhança, e vice-versa.

> **🔷 Conceito**
>
> Falta um último ajuste, e ele é de sinal. **O gradiente descendente minimiza**; a máquina construída no [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/index.html) sabe descer, não subir. E maximizar uma função é exatamente o mesmo que minimizar o negativo dela.
>
> Por isso o que vamos otimizar se chama **log-verossimilhança negativa**. Não é uma quantidade nova: é a log-verossimilhança da [seção 11.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/03-maxima-verossimilhanca.html) com o sinal trocado, para que o otimizador que já temos possa ser usado sem modificação. Fora deste livro você vai encontrá-la com outro nome — *log loss*, ou *binary cross-entropy* —, que é a mesma função sob rótulos diferentes.

### A perda e o seu gradiente

Para um único ponto:

In [ ]:
from scratch.linear_algebra import Vector, dot

def _negative_log_likelihood(x: Vector, y: float, beta: Vector) -> float:
    """A log-verossimilhança negativa de um único ponto"""
    if y == 1:
        return -math.log(logistic(dot(x, beta)))
    else:
        return -math.log(1 - logistic(dot(x, beta)))

Supondo que os pontos são independentes entre si, a verossimilhança do conjunto é o **produto** das verossimilhanças individuais — e portanto a log-verossimilhança é a **soma** dos logs:

In [ ]:
from typing import List

def negative_log_likelihood(xs: List[Vector],
                            ys: List[float],
                            beta: Vector) -> float:
    return sum(_negative_log_likelihood(x, y, beta)
               for x, y in zip(xs, ys))

Um pouco de cálculo dá o gradiente:

In [ ]:
from scratch.linear_algebra import vector_sum

def _negative_log_partial_j(x: Vector, y: float, beta: Vector, j: int) -> float:
    """A j-ésima derivada parcial, para um único ponto"""
    return -(y - logistic(dot(x, beta))) * x[j]

def _negative_log_gradient(x: Vector, y: float, beta: Vector) -> Vector:
    """O gradiente, para um único ponto"""
    return [_negative_log_partial_j(x, y, beta, j)
            for j in range(len(beta))]

def negative_log_gradient(xs: List[Vector],
                          ys: List[float],
                          beta: Vector) -> Vector:
    return vector_sum([_negative_log_gradient(x, y, beta)
                       for x, y in zip(xs, ys)])

> **🟩 Exemplo**
>
> "Um pouco de cálculo" é uma frase que costuma esconder trabalho. Aqui não esconde muito, e vale fazer a conta — é ela que explica por que `logistic_prime` foi definida lá em cima.
>
> Escreva $f$ para $f(\mathbf{x} \cdot \beta)$. A perda de um ponto é
>
> $$
> -\log L = -y \log f - (1 - y)\log(1 - f)
> $$
>
> Derivando em relação a $\beta_j$: a derivada de $\mathbf{x} \cdot \beta$ em relação a $\beta_j$ é $x_j$ (o mesmo argumento da [seção 12.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/03-ajustando-o-modelo.html)), e a derivada de $f$ é $f(1-f)$ — a propriedade conveniente. Pela regra da cadeia:
>
> $$
> \frac{\partial}{\partial \beta_j}(-\log L) = -y\,\frac{f(1-f)}{f}\,x_j + (1-y)\,\frac{f(1-f)}{1-f}\,x_j
> $$
>
> Os cancelamentos são o ponto: o $f$ do denominador do primeiro termo mata o $f$ do numerador, e o $(1-f)$ do segundo faz o mesmo. Sobra
>
> $$
> \bigl[-y(1-f) + (1-y)f\bigr] x_j = (f - y)\, x_j
> $$
>
> que é exatamente `-(y - logistic(dot(x, beta))) * x[j]`.
>
> Agora compare com o gradiente da regressão múltipla, `2 * err * x_j`, onde `err` era `previsão - y`. **É a mesma forma:** resíduo vezes $x_j$. O que mudou foi o significado de "previsão" — antes um número qualquer, agora uma probabilidade — e o fator 2 desapareceu no caminho. Se a estrutura do gradiente parece familiar, é porque ela é.

### Uma armadilha que ainda vai explodir

Olhe de novo o `_negative_log_likelihood`. Quando `y` é 0, ele calcula `math.log(1 - logistic(dot(x, beta)))`. Isso pressupõe que `logistic` nunca devolva exatamente `1.0` — porque `log(0)` não existe.

Matematicamente, a pressuposição está certa: a função logística é estritamente menor que 1 para todo $x$ finito. Em `float64`, está errada:

In [ ]:
for x in [10, 20, 30, 35, 36, 37, 40]:
    print(f"logistic({x:>2}) = {logistic(x)!r:<22} 1 - logistic({x:>2}) = {1 - logistic(x)!r}")

> **⚠️ Atenção — `logistic(37)` é `1.0`, e isso não é arredondamento de exibição**
>
> A partir de $x \approx 36{,}74$, `logistic(x)` devolve **exatamente** `1.0` — o `repr` acima mostra que não há dígitos escondidos. A razão é aritmética de ponto flutuante: `math.exp(-37)` vale cerca de $8{,}5 \times 10^{-17}$, e somar isso a 1 em `float64` não muda nada. O menor incremento representável acima de 1 é $2^{-52} \approx 2{,}2 \times 10^{-16}$, e uma parcela que não passa da **metade** disso ($1{,}11 \times 10^{-16}$) desaparece no arredondamento. O denominador vira `1.0` exato, e `1/1.0` é `1.0`.
>
> Dá para escrever o limiar exato. A saturação acontece quando `math.exp(-x)` fica menor ou igual a $2^{-53}$, e portanto quando
>
> $$
> x \;\geq\; -\log\bigl(2^{-53}\bigr) \;=\; 53 \ln 2 \;\approx\; 36{,}7368
> $$
>
> que é o 36,74 medido acima, agora até o último dígito — conferido por bissecção sobre a própria `logistic`. Não é um valor mágico da função logística; é a mantissa do `float64` aparecendo, com o sinal trocado por causa do `-x` no expoente.
>
> Consequência direta: `1 - logistic(37)` é `0.0`, e `math.log(0.0)` levanta `ValueError: math domain error`.
>
> Isso não é uma curiosidade sobre ponto flutuante. É uma bomba armada no código que acabamos de escrever, e ela **vai** explodir na próxima seção.
>
> O outro extremo tem um limite parecido e menos gentil: para $x$ muito negativo, `math.exp(-x)` estoura o alcance do `float64` e `logistic` levanta `OverflowError` em vez de devolver zero. O `logistic(-800)` deste código não devolve `0.0`; ele quebra.

> **💡 Dica — Na prática: `scipy` e `scikit-learn`**
>
> Nenhuma biblioteca séria calcula `log(1 - sigmoid(x))` do jeito que acabamos de escrever, e vale ver as duas saídas que elas usam.
>
> A primeira é ter uma versão da logística que não estoura. O `scipy` traz `scipy.special.expit`, que é a mesma função com um nome antigo (*expit* é o inverso do *logit*), implementada em C:
>
> ```python
> from scipy.special import expit
>
> expit(800)     # 1.0
> expit(-800)    # 0.0, sem OverflowError
> ```
>
> Repare que `expit(-800)` devolve `0.0` em vez de quebrar — melhor que o nosso, mas ainda saturado. Trocar `logistic` por `expit` **não** resolveria o problema desta seção: `expit(37)` também é `1.0` exato, e `log(1 - 1.0)` continua sendo `log(0)`.
>
> A segunda saída é a que realmente resolve, e é conceitual: **nunca calcular a probabilidade para depois tirar o log**. O `scipy` tem `log_expit`, que devolve $\log f(x)$ direto, sem passar pelo número intermediário:
>
> ```python
> from scipy.special import log_expit
>
> log_expit(-800)    # -800.0, exato
> ```
>
> `math.log(expit(-800))` levantaria `ValueError`; `log_expit(-800)` devolve `-800.0` sem esforço. É a mesma ideia da [seção 11.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/03-maxima-verossimilhanca.html), onde a log-verossimilhança foi calculada como soma de logs em vez de log de um produto: o número minúsculo nunca chega a existir na memória.
>
> Há ainda uma terceira postura, mais bruta, e é a que a métrica pronta do `scikit-learn` adota. `sklearn.metrics.log_loss` **corta** as probabilidades para dentro de $[\epsilon, 1-\epsilon]$ antes de tirar o log, com $\epsilon$ igual ao épsilon de máquina. Ela nunca levanta exceção: dar a ela uma probabilidade de `1.0` para um rótulo `0` devolve cerca de 36,04, que é $-\log(2{,}2 \times 10^{-16})$. É um número grande, sinalizando um erro grave, mas é um número — e é isso que ela devolve em vez do `ValueError` que o nosso código vai produzir na próxima seção.
>
> As três abordagens são defensáveis. A que **não** é defensável é a nossa, e ela está aqui de propósito: o erro que você vai ver na próxima seção é o que essas bibliotecas gastam código para esconder de você.

## Aplicando o Modelo

> **📌 Nota**
>
> Esta seção corresponde a *Applying the Model*, do capítulo 16 de Grus (2019).

Temos todas as peças: os dados, a função logística, a perda e o seu gradiente. Falta ajustar — e é aqui que este capítulo se separa dos dois anteriores.

> **❗ Importante — Não existe fórmula fechada para esta**
>
> A [seção 11.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/01-o-modelo.html) ajustou uma reta com duas médias, uma covariância e uma variância — sem laço nenhum. A [seção 12.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/03-ajustando-o-modelo.html) usou gradiente descendente para a regressão múltipla, mas deixou registrado que **a solução exata existia**: bastaria resolver um sistema linear, o que aquele capítulo não fez por decisão de escopo, não por impossibilidade.
>
> Aqui a situação muda de natureza. A condição de otimalidade da regressão logística — o gradiente igualado a zero — envolve $\beta$ **dentro** de uma exponencial, e não há manipulação algébrica que isole $\beta$ de lá. Não é uma conta pesada demais para caber num chunk: é uma conta que não existe. A [seção 5.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/05-ajustando-modelos.html) prometeu que este dia chegaria, e chegou. Para este modelo — e para as redes neurais dos capítulos [15](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/index.html) e [16](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/index.html) —, o gradiente descendente deixa de ser a alternativa prática e passa a ser o único caminho até um ajuste.

In [ ]:
from scratch.statistics import standard_deviation
import matplotlib.pyplot as plt
plt.close('all')

In [ ]:
from scratch.logistic_regression import (
    xs, ys, logistic, negative_log_likelihood, negative_log_gradient,
)

len(xs), xs[0], ys[0]

> **📌 Nota — A regra do kernel, valendo para o resto do capítulo**
>
> Cada página deste livro roda num kernel próprio, então as funções escritas na [seção anterior](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/02-a-funcao-logistica.html) não existem aqui. `scratch/logistic_regression.py` traz exatamente o mesmo código — é o módulo do livro-texto, copiado literalmente —, junto com os dados. Importar dali não é atalho: é o mesmo `logistic`, o mesmo `negative_log_likelihood` e o mesmo `negative_log_gradient` que você acabou de ver serem construídos.
>
> A mesma regra explica a repetição que vem pela frente. Funções importam-se de um módulo; **valores calculados, não** — e `rescale`, `beta` e a divisão treino/teste são valores. Por isso as seções [13.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/04-qualidade-do-ajuste.html) e [13.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/05-maquinas-de-vetores-de-suporte.html) refazem, cada uma, o `rescale` e os 5.000 epochs deste ajuste, com a mesma semente e os mesmos hiperparâmetros, para chegarem ao mesmo `beta` que sai aqui. É repetição de encanamento, não de conteúdo: lá o código vem escondido, e as duas seções apontam de volta para este callout.

### O que acontece se você simplesmente ajustar

O gradiente descendente precisa de um ponto de partida. Como sempre, um chute aleatório, com semente fixa:

In [ ]:
import random

random.seed(0)
beta = [random.random() for _ in range(3)]
beta

Antes de gastar 5.000 passos, vale avaliar a perda uma vez, nesse ponto de partida. É a conta mais barata do capítulo: `negative_log_likelihood(xs, ys, beta)`.

In [ ]:
try:
    perda = negative_log_likelihood(xs, ys, beta)
    print(f"perda inicial: {perda}")
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

Não houve ajuste ruim, convergência lenta nem coeficiente estranho. **O programa quebrou antes do primeiro passo.** Vale rastrear exatamente onde.

In [ ]:
from scratch.linear_algebra import dot

for i in [0, 1]:
    d = dot(xs[i], beta)
    print(f"ponto {i}: x = {xs[i]}, y = {ys[i]}")
    print(f"   dot(x, beta) = {d:.2f}")
    print(f"   logistic(...) = {logistic(d)!r}")
    print(f"   1 - logistic(...) = {1 - logistic(d)!r}")

O culpado é o **salário**. O primeiro usuário tem 48.000, e o coeficiente inicial sorteado para essa coluna foi 0,42 — o produto escalar dá 20.188,81, dominado inteiramente por aquela parcela. A experiência, que vale 0,7, contribui com 0,53; o termo constante, com 0,84. As outras duas colunas simplesmente não participam da conta.

E `logistic(20188.81)` é `1.0`, exato, pela razão que a [seção anterior](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/02-a-funcao-logistica.html) mediu: qualquer entrada acima de 36,74 satura.

Repare no que acontece com os dois primeiros pontos, porque eles falham de maneiras diferentes:

In [ ]:
from scratch.logistic_regression import _negative_log_likelihood

for i in [0, 1]:
    try:
        print(f"ponto {i} (y = {ys[i]}): perda = {_negative_log_likelihood(xs[i], ys[i], beta)!r}")
    except ValueError as e:
        print(f"ponto {i} (y = {ys[i]}): {type(e).__name__}: {e}")

> **⚠️ Atenção — Duas falhas, e a silenciosa é a pior**
>
> O ponto 0 tem `y = 1`, então a perda dele é `-math.log(logistic(...))`, que é `-log(1.0)`, que é **`-0.0`**. Nenhum erro: o modelo declara certeza absoluta de que este usuário paga, e a perda registra zero — perfeição — para um $\beta$ que foi **sorteado ao acaso segundos atrás**. Se todos os rótulos fossem 1, este ajuste reportaria perda zero e ninguém veria problema nenhum.
>
> O ponto 1 tem `y = 0`. A perda dele é `-math.log(1 - logistic(...))`, e `1 - 1.0` é `0.0`. Aí sim: `ValueError: math domain error`.
>
> A falha barulhenta é a sorte deste conjunto de dados — 148 dos 200 usuários têm `y = 0`, então o erro aparece no segundo ponto e é impossível de ignorar. A falha silenciosa é a que deveria assustar mais: ela não interrompe nada e produz um número que parece ótimo.

Vale parar aqui, porque este é o ponto do capítulo que sobrevive depois que o assunto "regressão logística" for esquecido.

`LogisticRegression().fit(X, y)` não teria produzido **nenhuma** das duas falhas. Nem o `ValueError`: a biblioteca calcula a perda por rotinas numericamente estáveis, que nunca chegam a formar `1 - 1.0`. Nem o `-0.0`: ela sequer expõe a perda de um ponto isolado — devolve coeficientes, previsões e um `score`, e mais nada. Os dois números que você acabou de ver **só são visíveis de dentro**. De fora, o mesmo ajuste teria rodado até o fim, sem uma linha de aviso, e devolvido um vetor de coeficientes com cara de resposta.

Generalize, porque a lição não é sobre a função logística: **o número na sua tela pode ser mentira, e a tela não tem como avisar.** Uma perda de zero é o que você esperaria de um modelo perfeito e é também o que sai de um $\beta$ sorteado ao acaso quando a aritmética satura. Os dois casos produzem o mesmo `0.0`, indistinguíveis. Quem só chama a função não tem de onde tirar a diferença; quem sabe que a logística tem um teto em `float64` e que a perda vira `-log(1.0)` sabe onde olhar. É por isso que esta disciplina constrói o código do zero, e é o motivo de esta seção ser a mais importante do capítulo.

### O conserto

O conserto é o mesmo passo que a [seção 13.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/01-o-problema.html) já tinha dado por higiene, e que aqui é a diferença entre existir um modelo e o programa quebrar: **reescalonar**. A função vem da [seção 7.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/06-reescalonamento.html) e é escrita em linha, porque o módulo onde ela mora não pode ser importado neste livro:

In [ ]:
from typing import List, Tuple
from scratch.linear_algebra import Vector, vector_mean
from scratch.statistics import standard_deviation

def scale(data: List[Vector]) -> Tuple[Vector, Vector]:
    """devolve a média e o desvio padrão de cada posição"""
    dim = len(data[0])
    means = vector_mean(data)
    stdevs = [standard_deviation([vector[i] for vector in data])
              for i in range(dim)]
    return means, stdevs

def rescale(data: List[Vector]) -> List[Vector]:
    """cada posição passa a ter média 0 e desvio padrão 1"""
    dim = len(data[0])
    means, stdevs = scale(data)
    rescaled = [v[:] for v in data]
    for v in rescaled:
        for i in range(dim):
            if stdevs[i] > 0:
                v[i] = (v[i] - means[i]) / stdevs[i]
    return rescaled

rescaled_xs = rescale(xs)
[round(v, 4) for v in rescaled_xs[0]]

O mesmo primeiro ponto, o mesmo $\beta$ sorteado, agora sem drama:

In [ ]:
d = dot(rescaled_xs[0], beta)
print(f"dot(x, beta) = {d:.4f}")
print(f"logistic(...) = {logistic(d):.4f}")
print(f"perda no conjunto todo = {negative_log_likelihood(rescaled_xs, ys, beta):.4f}")

> **🔷 Conceito**
>
> De 20.188,81 para −0,8059. Nada mudou no modelo, no otimizador ou nos dados — a mesma pessoa, com a mesma experiência e o mesmo salário, continua ali. O que mudou foi a **unidade**: salário deixou de ser medido em reais e passou a ser medido em desvios padrão a partir da média.
>
> Vale enunciar por que a diferença é tão violenta aqui e era só incômoda na regressão linear. Um modelo linear não tem teto: se o produto escalar der 20.000, a previsão é 20.000 — um número absurdo, mas um número, e o gradiente ainda aponta para algum lugar. A logística tem teto, e o alcance útil dela é aproximadamente $[-37, 37]$. Fora dessa janela, a função é **constante** em `float64`: derivada zero, gradiente zero, nenhuma informação sobre para onde ir. Reescalonar não é uma otimização de desempenho; é o que coloca os dados dentro da faixa em que a função ainda tem inclinação.

### O ajuste

Separamos treino e teste com o `train_test_split` do [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/index.html) e descemos o gradiente:

In [ ]:
from scratch.machine_learning import train_test_split

random.seed(0)
x_train, x_test, y_train, y_test = train_test_split(rescaled_xs, ys, 0.33)

len(x_train), len(x_test)

134 pontos de treino, 66 de teste. Agora o laço, que é o gradiente descendente em lote completo do [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/03-usando-o-gradiente.html) — sem minibatches, porque com 134 pontos e três parâmetros não há razão para complicar:

In [ ]:
import tqdm
from scratch.gradient_descent import gradient_step

learning_rate = 0.01

# ponto de partida aleatório, sorteado depois da divisão
beta = [random.random() for _ in range(3)]
perdas = []
betas = []          # guardamos o caminho inteiro; ele reaparece mais abaixo

with tqdm.trange(5000) as t:
    for epoch in t:
        gradient = negative_log_gradient(x_train, y_train, beta)
        beta = gradient_step(beta, gradient, -learning_rate)
        loss = negative_log_likelihood(x_train, y_train, beta)
        perdas.append(loss)
        betas.append(beta)
        t.set_description(f"loss: {loss:.3f} beta: {beta}")

[round(b, 4) for b in beta], round(perdas[-1], 4)

O $\beta$ ajustado é aproximadamente `[-2.02, 4.69, -4.47]`, com perda final de 39,96 no conjunto de treino. Vale ver como a perda chegou lá:

In [ ]:
# Figura: Log-verossimilhança negativa no conjunto de treino, por epoch (escala logarítmica no eixo x)
plt.plot(range(1, len(perdas) + 1), perdas)
plt.xscale('log')
plt.xlabel("epoch (escala logarítmica)")
plt.ylabel("log-verossimilhança negativa")
plt.title("Convergência do ajuste")
plt.show()

In [ ]:
for e in [1, 10, 100, 1000, 5000]:
    print(f"epoch {e:>5}: perda = {perdas[e - 1]:.4f}")

A queda é quase toda no começo. Depois do primeiro epoch a perda está em 113,07; no décimo já caiu para 61,76; no centésimo, 41,22 — a menos de 4% do valor final. No milésimo o número já é 39,9635, e os 4.000 epochs restantes movem a perda da sexta casa decimal para baixo: de 39,96350015 para 39,96349522, uma diferença de $4{,}9 \times 10^{-6}$. Quem ainda se mexe na terceira casa é o $\beta$, de 4,6903 para 4,6930 na segunda coordenada. É o padrão que a [seção 5.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/03-usando-o-gradiente.html) descreveu — queda rápida no início, cada vez mais lenta perto do mínimo — e é por isso que 5.000 passos aqui são generosidade, não necessidade.

### Voltando às unidades originais

Os coeficientes acima valem para os dados reescalonados. Dá para convertê-los de volta às unidades do mundo, desfazendo a transformação:

In [ ]:
means, stdevs = scale(xs)

beta_unscaled = [(beta[0]
                  - beta[1] * means[1] / stdevs[1]
                  - beta[2] * means[2] / stdevs[2]),
                 beta[1] / stdevs[1],
                 beta[2] / stdevs[2]]

beta_unscaled

E dá para verificar que os dois vetores descrevem o **mesmo** modelo, comparando a verossimilhança que cada um atribui ao conjunto completo. É o que Grus (2019) faz, com um `assert`:

In [ ]:
assert (negative_log_likelihood(xs, ys, beta_unscaled) ==
        negative_log_likelihood(rescaled_xs, ys, beta))

negative_log_likelihood(xs, ys, beta_unscaled)

O `assert` passa. Antes de concluir qualquer coisa disso, vale rodar a mesma comparação com outros $\beta$ — os do meio do treino, que ficaram guardados em `betas`:

In [ ]:
import math

def desescalonar(b: Vector) -> Vector:
    """a mesma conversão de cima, para um beta qualquer"""
    return [b[0] - b[1] * means[1] / stdevs[1] - b[2] * means[2] / stdevs[2],
            b[1] / stdevs[1],
            b[2] / stdevs[2]]

def as_duas_perdas(b: Vector) -> Tuple[float, float]:
    """a perda no conjunto completo, calculada nas duas escalas"""
    return (negative_log_likelihood(xs, ys, desescalonar(b)),
            negative_log_likelihood(rescaled_xs, ys, b))

for e in [100, 500, 1000, 2000, 3000, 4000, 5000]:
    esquerda, direita = as_duas_perdas(betas[e - 1])
    print(f"epoch {e:>5}: {esquerda!r:<19} {direita!r:<19} "
          f"iguais? {esquerda == direita}")

pares = [as_duas_perdas(b) for b in betas]
print(f"\nnos 5.000 epochs deste treino, comparando as duas escalas:")
print(f"   ==            vale em {sum(1 for a, c in pares if a == c)}")
print(f"   math.isclose  vale em {sum(1 for a, c in pares if math.isclose(a, c))}")

> **⚠️ Atenção — O `assert` passa, e passa por sorte**
>
> Aquele `assert` compara dois `float` por **igualdade exata**, e passa — para este $\beta$. Com o $\beta$ do epoch 500 do mesmo treino, os mesmos 200 pontos e a mesma conversão de escala, ele estouraria: 57,67164046623943 de um lado contra 57,67164046623944 do outro. Ao longo dos 5.000 epochs ele passaria em menos da metade delas. **É cara ou coroa**, e o $\beta$ final do Grus caiu do lado da cara.
>
> O motivo é que as duas contas **não** se cancelam ponto a ponto. Tome o $\beta$ final e compare `dot(x, beta_unscaled)` com o produto escalar na escala reescalonada, usuário por usuário: só **8** dos 200 pares saem bit a bit idênticos. Os outros 192 diferem na última casa — nada maior que $4{,}4 \times 10^{-15}$, mas diferem. A conversão é linear no papel; em `float64` ela é uma sequência **diferente** de multiplicações e subtrações, e duas sequências com o mesmo valor exato não têm por que produzir o mesmo valor arredondado.
>
> A igualdade que se vê aparece só **na soma final**, e por arredondamento: somados 200 termos com erros de última casa, o total às vezes cai no mesmo `float64` e às vezes cai no vizinho. Quando cai no vizinho, o `assert` quebra — sem nada de errado com o modelo, com a conversão ou com os dados.
>
> Este é exatamente o construto que a [seção 10.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/04-testando-o-modelo.html) marcou como **erro instrutivo do próprio livro-texto** — e o `assert` daqui também é do livro-texto (`scratch/logistic_regression.py`, no bloco `__main__`). Lá o não determinismo vinha da ordem de iteração de um `set`; aqui vem da ordem das operações aritméticas. É a mesma lição com outra fachada: **`==` entre `float` é a pergunta errada.** A pergunta certa tem tolerância, e a linha de cima mostra o placar dela — `math.isclose` vale nos 5.000 epochs, sem exceção.
>
> Fica também um aviso sobre `assert` em geral. Este passou em toda renderização deste livro porque a semente é fixa e o $\beta$ final é sempre o mesmo; um `assert` que passa **sempre** e um `assert` que passa **por construção** são coisas diferentes, e só a segunda é um teste.

### Interpretando os coeficientes

Infelizmente, estes coeficientes não são tão fáceis de ler quanto os de uma regressão linear. Mantendo tudo o mais constante:

- um ano a mais de experiência **adiciona 1,65 à entrada** da função logística;
- dez mil a mais de salário **subtrai 2,88 da entrada** da função logística.

O impacto disso na *saída*, porém, depende de onde a entrada já estava. Se `dot(beta, x_i)` já é grande — probabilidade perto de 1 —, aumentá-lo ainda mais quase não muda nada; se está perto de zero, um empurrão pequeno muda bastante. É a curva em S da seção anterior aparecendo na interpretação: **o mesmo aumento no argumento vale coisas diferentes em pontos diferentes**.

In [ ]:
for exp, sal in [(2.0, 60000), (5.0, 66700), (8.0, 60000), (8.0, 100000)]:
    p = logistic(dot(beta_unscaled, [1.0, exp, sal]))
    print(f"{exp:4.1f} anos de experiência, salário {sal:>6} -> P(paga) = {p:.4f}")

O que dá para afirmar com segurança é o sinal: mantendo tudo o mais constante, **mais experiência aumenta** a chance de pagar, e **salário maior diminui**. O segundo sinal parece estranho até você lembrar do gráfico da [seção 13.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/01-o-problema.html), em que os pagantes se concentravam embaixo à direita. Com 8 anos de experiência, um salário de 60.000 dá 99,22% de chance de conta paga; o mesmo profissional ganhando 100.000 cai para 0,13%.

> **⚠️ Atenção**
>
> Nada disso é uma afirmação causal. O modelo não diz que aumentar o salário de alguém faria a pessoa cancelar a assinatura. Ele diz que, **nestes 200 usuários**, quem ganha mais tende a não assinar — e a explicação plausível (quem tem experiência e ganha pouco está procurando emprego) é uma história que nós contamos, não algo que o ajuste tenha demonstrado. É a mesma advertência da [seção 12.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/04-interpretando-o-modelo.html), e ela não fica mais fraca porque a saída agora é uma probabilidade.

> **💡 Dica — Na prática: `scikit-learn`**
>
> ```python
> from sklearn.linear_model import LogisticRegression
>
> X = [x[1:] for x in xs]          # sem a coluna de 1
> modelo = LogisticRegression().fit(X, ys)
>
> modelo.intercept_, modelo.coef_
> ```
>
> Três coisas que vale saber, agora que você viu o código por dentro.
>
> **A biblioteca não quebra com os dados brutos — e também não teria mostrado a falha silenciosa.** Rodar o código acima nos `xs` sem reescalonar funciona: ela devolve intercepto 8,41 e coeficientes 1,51 e −0,00027, na mesma vizinhança do nosso `beta_unscaled` — `[8.93, 1.65, -0.000288]`. Isso não contradiz nada do que esta seção mostrou; é o resultado de um trabalho de engenharia que a implementação faz e a nossa não. O otimizador padrão (`lbfgs`) usa curvatura de segunda ordem, e por isso não sofre com colunas de escalas diferentes do jeito que um passo fixo sofre; e a perda é calculada por rotinas numericamente estáveis, que nunca formam `1 - 1.0`. A saturação em `float64` continua existindo — ela é uma propriedade do tipo, não do código —, mas nenhuma linha do seu programa a encontra.
>
> Repare que isso vale para as **duas** falhas, e não só para a barulhenta. O `ValueError` some porque a perda é calculada de outro jeito; o `-0.0` some porque a interface não tem onde exibi-lo — `LogisticRegression` devolve coeficientes, previsões e um `score`, nunca a perda de um ponto isolado. Nenhum dos dois números do começo desta seção existe do lado de fora. O `ValueError` é, literalmente, o que você paga para ver o mecanismo; e o `-0.0` é o que você **só** vê se pagar.
>
> **Ela regulariza por padrão, e você não pediu.** `LogisticRegression` aplica penalidade L2 com `C=1.0` a menos que você diga o contrário — a mesma regularização *ridge* da [seção 12.8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/08-regularizacao.html), aqui ligada de fábrica. É por isso que os coeficientes dela saem um pouco encolhidos em relação aos nossos. Aumentar `C` afrouxa a penalidade; com ela praticamente desligada e os dados reescalonados, a biblioteca devolve `[-2.11, 4.53, -4.40]` contra os nossos `[-2.02, 4.69, -4.47]`, e a diferença que sobra é a distância entre um otimizador que converge de verdade e 5.000 passos de tamanho fixo.
>
> **Reescalonar continua sendo boa ideia mesmo assim** — não pelo `ValueError`, que a biblioteca não tem, mas pela penalidade. Uma penalidade que soma quadrados de coeficientes trata todas as colunas com o mesmo rigor, e um coeficiente medido em "por real de salário" é numericamente minúsculo perto de um medido em "por ano de experiência". Sem reescalonar, a regularização pune as duas colunas de forma desigual por um motivo que não tem nada a ver com o problema. O `StandardScaler` da [seção 7.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/06-reescalonamento.html) é a resposta.

## Qualidade do Ajuste

> **📌 Nota**
>
> Esta seção corresponde a *Goodness of Fit*, do capítulo 16 de Grus (2019).

Ainda não usamos os 66 pontos de teste que a seção anterior separou. Chegou a hora — e note que a pergunta desta seção é diferente da do capítulo passado. Lá, a qualidade do ajuste era o R², a fração da variação de `y` que o modelo capturava. Aqui `y` não tem variação a ser explicada: ele vale 0 ou 1. O que se mede é **quantas vezes o modelo acerta, e de que jeito ele erra** — o vocabulário do [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/04-correcao.html).

In [ ]:
from scratch.statistics import standard_deviation
import matplotlib.pyplot as plt
plt.close('all')

> **📌 Nota**
>
> De novo o de sempre: kernel próprio, `beta` não atravessa páginas — é a regra que o callout da [seção 13.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/03-aplicando-o-modelo.html) enunciou. O bloco a seguir vem escondido porque não é a lição desta seção: ele redefine `scale` e `rescale`, refaz a divisão treino/teste com `random.seed(0)` e roda os mesmos 5.000 epochs com taxa 0,01. A saída é o `beta` de lá, coordenada por coordenada, e é ele que o resto desta seção usa.

In [ ]:
import random
from typing import List, Tuple
from scratch.linear_algebra import Vector, dot, vector_mean
from scratch.logistic_regression import (
    xs, ys, logistic, negative_log_likelihood, negative_log_gradient,
)
from scratch.gradient_descent import gradient_step
from scratch.machine_learning import train_test_split

def scale(data: List[Vector]) -> Tuple[Vector, Vector]:
    dim = len(data[0])
    return (vector_mean(data),
            [standard_deviation([v[i] for v in data]) for i in range(dim)])

def rescale(data: List[Vector]) -> List[Vector]:
    dim = len(data[0])
    means, stdevs = scale(data)
    rescaled = [v[:] for v in data]
    for v in rescaled:
        for i in range(dim):
            if stdevs[i] > 0:
                v[i] = (v[i] - means[i]) / stdevs[i]
    return rescaled

rescaled_xs = rescale(xs)

random.seed(0)
x_train, x_test, y_train, y_test = train_test_split(rescaled_xs, ys, 0.33)

beta = [random.random() for _ in range(3)]
for _ in range(5000):
    gradient = negative_log_gradient(x_train, y_train, beta)
    beta = gradient_step(beta, gradient, -0.01)

[round(b, 4) for b in beta]

### As quatro caixas

O modelo devolve uma **probabilidade**, não um rótulo. Para contar acertos é preciso primeiro transformá-la numa decisão, e a regra mais simples é a mais óbvia: prever "conta paga" sempre que a probabilidade passar de 0,5.

In [ ]:
true_positives = false_positives = true_negatives = false_negatives = 0

for x_i, y_i in zip(x_test, y_test):
    prediction = logistic(dot(beta, x_i))

    if y_i == 1 and prediction >= 0.5:   # VP: paga e previmos que paga
        true_positives += 1
    elif y_i == 1:                       # FN: paga e previmos que não paga
        false_negatives += 1
    elif prediction >= 0.5:              # FP: não paga e previmos que paga
        false_positives += 1
    else:                                # VN: não paga e previmos que não paga
        true_negatives += 1

true_positives, false_positives, false_negatives, true_negatives

Doze verdadeiros positivos, quatro falsos positivos, três falsos negativos e 47 verdadeiros negativos. Em forma de tabela:

|  | Paga | Não paga |
|---|---|---|
| **Previu "paga"** | 12 | 4 |
| **Previu "não paga"** | 3 | 47 |

E as métricas do Capítulo 8, importadas de onde foram escritas:

In [ ]:
from scratch.machine_learning import accuracy, precision, recall, f1_score

tp, fp = true_positives, false_positives
fn, tn = false_negatives, true_negatives

print(f"acurácia:  {accuracy(tp, fp, fn, tn):.4f}")
print(f"precisão:  {precision(tp, fp, fn, tn):.4f}")
print(f"revocação: {recall(tp, fp, fn, tn):.4f}")
print(f"F1:        {f1_score(tp, fp, fn, tn):.4f}")

**Precisão de 0,75**: quando prevemos "conta paga", acertamos em 75% das vezes. **Revocação de 0,8**: dos usuários que de fato pagam, encontramos 80%. Nada mau, para um conjunto deste tamanho.

> **⚠️ Atenção — A acurácia, de novo, é a métrica que menos diz**
>
> A acurácia deu 0,894, e é o número mais alto dos quatro — o que deveria acender a luz que o [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/04-correcao.html) instalou.
>
> Só 15 dos 66 usuários do conjunto de teste pagam. Um classificador que respondesse "não paga" para todo mundo, sem olhar dado nenhum, acertaria 51 dos 66 — **acurácia de 0,773**, a poucos pontos do nosso modelo, com revocação exatamente zero e precisão indefinida (o mesmo `ZeroDivisionError` que o [Capítulo 10](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/05-usando-o-modelo.html) exibiu).
>
> O que separa o nosso modelo daquele não aparece na acurácia. Aparece na revocação: 0,8 contra 0,0.

### O limiar de 0,5 é uma escolha, não uma propriedade

A comparação `prediction >= 0.5` está no meio daquele laço como se fosse parte do algoritmo. Não é. O modelo produz probabilidades; **0,5 é uma decisão de quem usa o modelo**, e trocá-la troca todas as métricas:

In [ ]:
print(f"{'limiar':>7} {'VP':>4} {'FP':>4} {'FN':>4} {'VN':>4} {'precisão':>10} {'revocação':>11}")
previsoes = [logistic(dot(beta, x_i)) for x_i in x_test]

for limiar in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    tp = sum(1 for p, y in zip(previsoes, y_test) if y == 1 and p >= limiar)
    fp = sum(1 for p, y in zip(previsoes, y_test) if y == 0 and p >= limiar)
    fn = sum(1 for p, y in zip(previsoes, y_test) if y == 1 and p < limiar)
    tn = sum(1 for p, y in zip(previsoes, y_test) if y == 0 and p < limiar)
    print(f"{limiar:>7.1f} {tp:>4} {fp:>4} {fn:>4} {tn:>4} "
          f"{tp / (tp + fp):>10.4f} {tp / (tp + fn):>11.4f}")

> **🔷 Conceito**
>
> A tabela é o compromisso da [seção 8.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/04-correcao.html) instanciado num modelo real. Baixar o limiar para 0,2 leva a revocação a 0,867 — o modelo passa a encontrar 13 dos 15 pagantes — ao custo de derrubar a precisão para 0,565, porque agora ele acusa 10 pessoas que não pagam. Subir para 0,8 inverte tudo: precisão de 0,889, revocação de 0,533.
>
> **O modelo é o mesmo em todas as linhas.** O `beta` não mudou; nem um epoch de gradiente descendente foi rodado entre uma linha e a seguinte. O que muda é onde se corta a probabilidade, e essa escolha depende do custo de cada tipo de erro — que vem do problema, não dos dados. Se o objetivo é uma campanha de marketing barata para converter usuários, um falso positivo custa um e-mail e vale a pena baixar o limiar. Se o objetivo é dar desconto para quem já ia pagar, cada falso positivo é dinheiro perdido, e o limiar sobe.

### Previsto contra real

Dá também para desenhar as previsões contra os valores reais:

In [ ]:
# Figura: Regressão logística: probabilidade prevista contra o resultado real
plt.scatter(previsoes, y_test, marker='+')
plt.axvline(0.5, color='gray', linestyle=':')
plt.xlabel("probabilidade prevista")
plt.ylabel("resultado real")
plt.yticks([0, 1])
plt.title("Regressão logística: previsto contra real")
plt.show()

Compare com a figura equivalente da [seção 13.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/01-o-problema.html), a do modelo linear. Duas diferenças importam. A primeira: aqui **todo ponto está entre 0 e 1** — não há mais probabilidade negativa, porque a função logística tornou isso impossível, não improvável. A segunda: as duas fileiras não se sobrepõem do mesmo jeito. Os usuários que não pagam se amontoam à esquerda, com probabilidades quase nulas; os que pagam se espalham pela metade direita. Mas a separação não é limpa, e o que a estraga não está no meio do gráfico: **três pagantes caem à esquerda da linha pontilhada**, e só um deles é limítrofe — os outros dois aparecem em 0,163 e 0,024, lá na ponta em que o modelo não hesitou, apenas errou. Perto do limiar ficam os casos difíceis; nas pontas ficam os erros caros.

In [ ]:
erros = [(p, y) for p, y in zip(previsoes, y_test)
         if (p >= 0.5) != (y == 1)]

print(f"{len(erros)} erros em {len(y_test)} pontos de teste")
for p, y in sorted(erros):
    print(f"   probabilidade prevista {p:.4f}, resultado real {y}")

Sete erros, e eles não são todos do mesmo tipo. Os quatro falsos positivos vão de 0,535 a 0,845 — o modelo apostou "paga" com confiança que vai da dúvida à convicção, e perdeu nas quatro. Dos três falsos negativos, um é limítrofe: com 0,457 o modelo quase acertou, e é o tipo de erro que se aceita. Outro, com 0,163, já é uma aposta errada com alguma convicção — 84% de confiança do lado errado.

O sétimo é diferente. É um usuário que **paga**, e a quem o modelo atribuiu probabilidade **0,0235** — ou seja, declarou com 97,6% de confiança que não pagaria. Esse não é um erro de calibração de limiar; nenhum limiar entre 0 e 1 o corrigiria sem destruir o resto. É um ponto que está do lado errado da fronteira que o modelo traçou, e nenhuma reta traçada naquele plano conseguiria colocá-lo do lado certo sem levar junto uma porção de não pagantes. Que fronteira é essa, e o que significa não existir uma que separe as classes perfeitamente, é o assunto da [próxima seção](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/05-maquinas-de-vetores-de-suporte.html).

> **💡 Dica — Na prática: `scikit-learn`**
>
> Tudo o que esta seção calculou à mão cabe em três linhas:
>
> ```python
> from sklearn.metrics import classification_report, confusion_matrix
>
> previsto = [1 if logistic(dot(beta, x)) >= 0.5 else 0 for x in x_test]
> print(confusion_matrix(y_test, previsto))
> print(classification_report(y_test, previsto))
> ```
>
> `classification_report` devolve precisão, revocação e F1 **para cada classe**, e não só para a classe positiva — o que é uma correção útil ao hábito de olhar só o lado que interessa.
>
> Sobre o limiar, há uma armadilha de interface que vale conhecer. Um modelo do `scikit-learn` tem dois métodos:
>
> - `modelo.predict(X)` devolve rótulos, e embute o limiar de 0,5 sem perguntar;
> - `modelo.predict_proba(X)` devolve as probabilidades, e deixa a decisão com você.
>
> Quem usa só o `predict` nunca vê que houve uma escolha — e é exatamente a escolha que esta seção mostrou valer a diferença entre revocação 0,53 e 0,87. Para explorar o compromisso inteiro de uma vez, `sklearn.metrics.precision_recall_curve` percorre todos os limiares possíveis, e `roc_curve` faz o mesmo do ponto de vista da taxa de falsos positivos.

## Máquinas de Vetores de Suporte

> **📌 Nota**
>
> Esta seção corresponde a *Support Vector Machines*, do capítulo 16 de Grus (2019).

O conjunto de pontos onde `dot(beta, x_i)` vale exatamente 0 é a **fronteira** entre as nossas duas classes: ali a função logística devolve 0,5, e a decisão vira cara ou coroa. De um lado prevemos "paga", do outro "não paga". Vale desenhar essa fronteira para ver com clareza o que o modelo está fazendo.

In [ ]:
from scratch.statistics import standard_deviation
import matplotlib.pyplot as plt
plt.close('all')

> **📌 Nota**
>
> De novo o de sempre: kernel próprio, `beta` não atravessa páginas — a regra está no callout da [seção 13.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/03-aplicando-o-modelo.html). O bloco a seguir vem escondido pelo mesmo motivo da [seção 13.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/04-qualidade-do-ajuste.html): ele refaz o `rescale`, a divisão treino/teste e os mesmos 5.000 epochs daquela seção, e no fim desfaz a escala. A saída é o `beta_unscaled` da seção 13.3 — os coeficientes em anos e em reais, que é a forma de que precisamos para desenhar a fronteira no plano do gráfico.

In [ ]:
import random
from typing import List, Tuple
from scratch.linear_algebra import Vector, dot, vector_mean
from scratch.logistic_regression import xs, ys, negative_log_gradient
from scratch.gradient_descent import gradient_step
from scratch.machine_learning import train_test_split

def scale(data: List[Vector]) -> Tuple[Vector, Vector]:
    dim = len(data[0])
    return (vector_mean(data),
            [standard_deviation([v[i] for v in data]) for i in range(dim)])

def rescale(data: List[Vector]) -> List[Vector]:
    dim = len(data[0])
    means, stdevs = scale(data)
    rescaled = [v[:] for v in data]
    for v in rescaled:
        for i in range(dim):
            if stdevs[i] > 0:
                v[i] = (v[i] - means[i]) / stdevs[i]
    return rescaled

rescaled_xs = rescale(xs)

random.seed(0)
x_train, x_test, y_train, y_test = train_test_split(rescaled_xs, ys, 0.33)
beta = [random.random() for _ in range(3)]
for _ in range(5000):
    beta = gradient_step(beta, negative_log_gradient(x_train, y_train, beta), -0.01)

means, stdevs = scale(xs)
beta_unscaled = [(beta[0]
                  - beta[1] * means[1] / stdevs[1]
                  - beta[2] * means[2] / stdevs[2]),
                 beta[1] / stdevs[1],
                 beta[2] / stdevs[2]]

beta_unscaled

A fronteira é `beta_unscaled[0] + beta_unscaled[1] * experiência + beta_unscaled[2] * salário = 0`. Isolando o salário, ela vira uma reta no plano do gráfico da [seção 13.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/01-o-problema.html):

In [ ]:
# Figura: Usuários pagantes e não pagantes, com a fronteira de decisão
b0, b1, b2 = beta_unscaled

def salario_na_fronteira(experiencia: float) -> float:
    return -(b0 + b1 * experiencia) / b2

experiencia = [x[1] for x in xs]
salario = [x[2] for x in xs]

plt.scatter([e for e, y in zip(experiencia, ys) if y == 1],
            [s for s, y in zip(salario, ys) if y == 1],
            marker='+', label='paga')
plt.scatter([e for e, y in zip(experiencia, ys) if y == 0],
            [s for s, y in zip(salario, ys) if y == 0],
            marker='.', label='não paga')

grade = [0, 10]
plt.plot(grade, [salario_na_fronteira(e) for e in grade],
         color='black', label='fronteira de decisão')

plt.xlabel("anos de experiência")
plt.ylabel("salário")
plt.legend(loc='upper left')
plt.title("Fronteira de decisão")
plt.show()

In [ ]:
errados = sum(1 for x, y in zip(xs, ys)
              if (dot(beta_unscaled, x) >= 0) != (y == 1))

print(f"{errados} dos {len(xs)} usuários ficam do lado errado da fronteira")

A reta atravessa a nuvem na diagonal, subindo da esquerda para a direita: abaixo dela fica quase todo mundo que paga, acima quase todo mundo que não paga. É a região de "experiência alta, salário baixo" que a seção 13.1 já tinha identificado a olho nu, agora com um contorno. Vinte e um dos 200 usuários ficam do lado errado dela.

> **🔷 Conceito**
>
> Essa fronteira é um **hiperplano** que corta o **espaço de atributos** em dois semiespaços — em duas dimensões, uma reta; em três, um plano; em mais, algo que não dá para desenhar mas que se comporta igual.
>
> E note como ela apareceu: **de brinde**. Nós não pedimos uma fronteira; pedimos o $\beta$ que maximiza a verossimilhança dos dados sob o modelo logístico. O hiperplano é um efeito colateral desse pedido.
>
> A pergunta que abre esta seção é: e se a fronteira fosse o pedido principal?

> **⚠️ Atenção — Espaço de atributos, não espaço de parâmetros**
>
> Neste ponto Grus (2019) escreve que o hiperplano corta o *espaço de parâmetros*. É um deslize, e vale desfazê-lo com cuidado, porque os dois espaços têm três dimensões neste problema e trocá-los é fácil.
>
> - O **espaço de atributos** é onde os dados moram: cada ponto é um usuário, $(1, \text{experiência}, \text{salário})$. É o plano da figura logo acima. A fronteira é o conjunto $\{\mathbf{x} : \beta \cdot \mathbf{x} = 0\}$ — um conjunto de **x**, portanto um objeto deste espaço.
> - O **espaço de parâmetros** é onde mora o $\beta$: cada ponto dele é um modelo candidato inteiro. Foi por ele que o gradiente descendente caminhou na [seção 13.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/03-aplicando-o-modelo.html) — os 5.000 epochs são uma trajetória aqui, não ali —, e é sobre ele que se desenha a superfície de perda do [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/index.html).
>
> O que provavelmente gera o deslize é uma simetria real: $\beta \cdot \mathbf{x} = 0$ define um hiperplano nos **dois** espaços, dependendo de qual dos dois vetores você segura fixo. Com $\beta$ fixo e $\mathbf{x}$ variando, sai a fronteira de decisão. Com um $\mathbf{x}$ fixo e $\beta$ variando, sai o conjunto de todos os modelos que consideram aquele usuário exatamente limítrofe — um objeto legítimo e útil, mas que não é o que está desenhado no gráfico.
>
> Este livro marca os erros do livro-texto em vez de reproduzi-los em silêncio — a [seção 10.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/04-testando-o-modelo.html) e a [seção 13.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/03-aplicando-o-modelo.html) fazem o mesmo com um `assert`. A diferença aqui é que o deslize é de vocabulário, não de código, o que não o torna inofensivo: quem embaralha os dois espaços não tem como entender por que multiplicar $\beta$ por 2 muda o modelo sem mover um milímetro da fronteira. É precisamente o que acontece no callout mais abaixo, sobre dados separáveis.

### O critério da margem

A ideia da **máquina de vetores de suporte** (*support vector machine*, ou SVM) é essa. Em vez de estimar probabilidades e deixar a fronteira aparecer no fim, ela procura diretamente o hiperplano que **melhor separa** as classes nos dados de treino — e "melhor" tem uma definição precisa: o hiperplano que **maximiza a distância até o ponto mais próximo de cada classe**. Essa distância se chama **margem**, e os pontos que a tocam são os **vetores de suporte** que dão nome ao método.

Por que isso é um critério, e não só uma preferência estética? Porque, quando as classes são separáveis, existem **infinitos** hiperplanos que classificam o treino sem erro nenhum, e eles não são equivalentes num conjunto de teste:

In [ ]:
# Figura: Quatro retas que separam perfeitamente o mesmo treino; só uma maximiza a margem
classe_a = [(-2.0, -1.0), (-1.0, -2.0), (-1.5, -1.5), (-2.5, -0.5), (-3.0, -1.8)]
classe_b = [(2.0, 1.0), (1.0, 2.0), (1.5, 1.5), (2.5, 0.5), (3.0, 1.8)]

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter([p[0] for p in classe_a], [p[1] for p in classe_a], marker='.', s=90)
ax.scatter([p[0] for p in classe_b], [p[1] for p in classe_b], marker='+', s=90)

grade = [-4.0, 4.0]

# três separadoras válidas, mas desnecessariamente próximas de algum ponto
for a, b in [(-1.0, -2.0), (-1.0, 2.0), (-0.5, -1.0)]:
    ax.plot(grade, [a * x + b for x in grade], ':', color='gray')

# a de margem máxima, com as duas linhas que a margem toca
ax.plot(grade, [-x for x in grade], '-', color='black', linewidth=2)
ax.plot(grade, [-x - 3 for x in grade], '--', color='black', linewidth=0.8)
ax.plot(grade, [-x + 3 for x in grade], '--', color='black', linewidth=0.8)

ax.set_xlim(-4, 4)
ax.set_ylim(-4, 4)
ax.set_title("Qual das retas separadoras é a melhor?")
plt.show()

As quatro retas acertam **todos** os pontos de treino: as três pontilhadas cinzas e a cheia preta classificam este conjunto sem um erro sequer. A diferença está na folga. A reta cheia é a que fica mais longe dos pontos mais próximos de cada lado, e as duas tracejadas finas marcam onde a margem dela encosta — nos quatro pontos de cada classe que são, por definição, os vetores de suporte. As pontilhadas passam a cerca de um terço dessa distância de algum ponto de treino.

A intuição é que um ponto novo, que caia um pouco fora do padrão do treino, tem mais chance de continuar do lado certo se a fronteira estiver o mais afastada possível de todo mundo.

> **❗ Importante — O critério da margem não é o critério da verossimilhança**
>
> Esta é a diferença que importa entre os dois métodos, e ela tem duas faces.
>
> **A primeira: quais pontos importam.** A log-verossimilhança negativa que a [seção 13.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/02-a-funcao-logistica.html) construiu soma uma parcela por ponto — **todos** os pontos, inclusive os que estão longe e do lado certo. Um usuário para quem o modelo prevê probabilidade 0,999 e que de fato paga ainda contribui com uma perda pequena, mas positiva, e o gradiente ainda tem um termo para ele. Mover esse usuário para ainda mais longe da fronteira **melhora** a verossimilhança e desloca o $\beta$ ajustado. Já a margem máxima depende **apenas** dos pontos mais próximos: mover um ponto distante não muda o hiperplano da SVM em coisa nenhuma, porque ele não é um vetor de suporte.
>
> **A segunda: em dados separáveis, a verossimilhança não tem máximo.** Este é o caso em que a diferença deixa de ser filosófica. Se existe um hiperplano que separa as classes perfeitamente, então multiplicar $\beta$ por 2 empurra todas as probabilidades corretas ainda mais para perto de 0 ou 1 e **reduz** a log-verossimilhança negativa. Multiplicar por 4 reduz mais. Não existe um $\beta$ ótimo: existe uma direção ótima e um comprimento que cresce para sempre.
>
> Dá para ver isso acontecendo — nos mesmos pontos da figura acima, menos o mais afastado de cada classe, para caber em duas linhas de código:

In [ ]:
import math
from scratch.logistic_regression import negative_log_likelihood

xs_sep = [[1.0, -2.0, -1.0], [1.0, -1.0, -2.0], [1.0, -1.5, -1.5], [1.0, -2.5, -0.5],
          [1.0,  2.0,  1.0], [1.0,  1.0,  2.0], [1.0,  1.5,  1.5], [1.0,  2.5,  0.5]]
ys_sep = [0, 0, 0, 0, 1, 1, 1, 1]

random.seed(0)
beta_sep = [random.random() for _ in range(3)]
marcos = [1, 10, 100, 1000, 10000, 50000]

for epoch in range(1, max(marcos) + 1):
    gradiente = negative_log_gradient(xs_sep, ys_sep, beta_sep)
    beta_sep = gradient_step(beta_sep, gradiente, -0.01)
    if epoch in marcos:
        norma = math.sqrt(dot(beta_sep, beta_sep))
        perda = negative_log_likelihood(xs_sep, ys_sep, beta_sep)
        print(f"epoch {epoch:>6}: |beta| = {norma:8.4f}   perda = {perda:.6f}")

> O comprimento de $\beta$ cresce sem parar e a perda desce em direção a zero sem nunca chegar. O gradiente descendente não "converge" aqui — ele apenas anda cada vez mais devagar, e para onde estava quando você mandou parar. **O modelo que você obtém é uma função de quantos epochs você teve paciência de rodar**, o que é uma forma desconfortável de escolher um modelo.
>
> É por isso que o `LogisticRegression` do `scikit-learn` liga a regularização de fábrica, como o callout da [seção 13.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/03-aplicando-o-modelo.html) mencionou: a penalidade sobre o tamanho de $\beta$ é o que impede essa fuga para o infinito. A máquina de vetores de suporte não precisa desse remendo — o critério de margem já é uma restrição sobre o tamanho do vetor, embutida na definição do problema.

### Quando não existe hiperplano separador

Encontrar o hiperplano de margem máxima é um problema de otimização com restrições, e as técnicas que o resolvem estão fora do alcance deste livro — é por isso que esta seção não implementa a máquina de vetores de suporte, e é a única do capítulo que deixa de implementar o seu próprio assunto.

Mas há um problema anterior e mais interessante: **um hiperplano separador pode simplesmente não existir**. É o caso destes dados. Não é uma impressão vinda do gráfico; dá para verificar por busca exaustiva.

In [ ]:
def separavel(pontos, rotulos) -> bool:
    """Existe alguma reta que separe perfeitamente as duas classes?"""
    n = len(pontos)
    for i in range(n):
        ax, ay = pontos[i]
        for j in range(i + 1, n):
            bx, by = pontos[j]
            dx, dy = bx - ax, by - ay
            # direção do segmento, e a perpendicular a ele
            for wx, wy in [(dx, dy), (dy, -dx)]:
                if wx == 0 and wy == 0:
                    continue
                proj0 = [wx * px + wy * py
                         for (px, py), r in zip(pontos, rotulos) if r == 0]
                proj1 = [wx * px + wy * py
                         for (px, py), r in zip(pontos, rotulos) if r == 1]
                if max(proj0) < min(proj1) or max(proj1) < min(proj0):
                    return True
    return False

pontos = [(x[1], x[2]) for x in xs]

if separavel(pontos, ys):
    print("Existe uma reta que separa pagantes de não pagantes.")
else:
    print("Nenhuma reta separa estes dados.")

> **📌 Nota**
>
> Por que testar só direções ligadas a **pares de pontos** basta, e a busca é exaustiva apesar de finita: se as duas classes são separáveis no plano, os fechos convexos delas são disjuntos, e o segmento mais curto entre os dois fechos define a direção de separação. Esse segmento ou liga dois vértices — um de cada classe, e a direção é a do próprio segmento — ou liga um vértice a uma aresta, e aí a direção é perpendicular a essa aresta, que por sua vez liga dois pontos da mesma classe. Percorrer todos os pares nas duas formas cobre os dois casos. São 19.900 pares e menos de um segundo de execução.

Não há reta nenhuma que separe pagantes de não pagantes neste conjunto. A SVM, na forma acima, não teria o que devolver.

### O truque do kernel

Às vezes dá para contornar isso **transformando os dados num espaço de dimensão maior**. Considere o conjunto unidimensional mais simples possível em que nenhum ponto de corte funciona:

In [ ]:
# Figura: À esquerda, um conjunto unidimensional não separável; à direita, o mesmo conjunto depois do mapa x → (x, x²)
valores = [-3.0, -2.5, -2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0, 2.5, 3.0]
rotulos = [1 if abs(v) >= 2 else 0 for v in valores]

fig, (esquerda, direita) = plt.subplots(1, 2, figsize=(10, 4))

for marcador, classe in [('+', 1), ('.', 0)]:
    selecionados = [v for v, r in zip(valores, rotulos) if r == classe]
    esquerda.scatter(selecionados, [0] * len(selecionados), marker=marcador, s=90)
    direita.scatter(selecionados, [v ** 2 for v in selecionados],
                    marker=marcador, s=90)

esquerda.set_yticks([])
esquerda.set_ylim(-1, 1)
esquerda.set_box_aspect(0.3)   # o painel da esquerda é uma reta, não um plano
esquerda.set_anchor("N")
esquerda.set_xlabel("x")
esquerda.set_title("Uma dimensão: nenhum corte separa")

direita.axhline(2.5, color='black')
direita.set_xlabel("x")
direita.set_ylabel("x²")
direita.set_title("Duas dimensões: uma reta separa")

plt.tight_layout()
plt.show()

À esquerda, uma classe ocupa as duas pontas da reta e a outra ocupa o meio. Não existe ponto de corte que resolva: qualquer corte único deixa exemplos da classe das pontas dos dois lados. À direita, o mesmo conjunto depois de enviar cada ponto $x$ para o par $(x, x^2)$. Nada foi inventado — a segunda coordenada é uma função da primeira, calculada a partir dela —, mas agora a reta horizontal $y = 2{,}5$ separa perfeitamente.

> **🔷 Conceito**
>
> Isso normalmente é chamado de **truque do kernel**, e o "truque" está numa economia. Mapear de fato todos os pontos para o espaço de dimensão maior pode ser caro — ou impossível, se o espaço tiver dimensão infinita, o que acontece com alguns kernels usados na prática.
>
> Acontece que o algoritmo da SVM só precisa dos **produtos escalares** entre pares de pontos, nunca das coordenadas individuais. Uma função de *kernel* calcula esses produtos escalares no espaço maior **direto a partir das coordenadas originais**, sem nunca construir os vetores transformados. Você ganha a fronteira curva sem pagar o custo da dimensão.
>
> É o que a SVM compra, e é a razão de ela ter dominado a classificação durante uma década, antes das redes neurais: **fronteiras não lineares com um problema de otimização que continua convexo** — logo, com solução única, sem mínimos locais e sem dependência do ponto de partida. Compare com o [Capítulo 15](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/index.html), onde a não linearidade vem de outro jeito e custa exatamente essas garantias.

Usar máquinas de vetores de suporte sem depender de software de otimização especializado, escrito por quem tem o preparo para isso, é difícil e provavelmente não é uma boa ideia — e é aqui que Grus (2019) encerra o tratamento do assunto. Este livro faz o mesmo: esta é a única seção do capítulo sem uma implementação do zero, e a razão não é falta de espaço. É que a implementação honesta seria um algoritmo de programação quadrática com restrições, e escrever um não ensinaria nada sobre classificação.

### O que este capítulo deixa

Três capítulos, uma escada, e o degrau de cima é este. A [seção 11.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/01-o-modelo.html) ajustou uma reta com uma **fórmula fechada** — duas médias, uma covariância, uma variância, sem laço nenhum. O [Capítulo 12](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/index.html) manteve a fórmula fechada existindo e parou de usá-la: a solução exata era um sistema linear que aquele capítulo escolheu não resolver, por escopo. Aqui ela não existe, e a promessa da [seção 5.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/05-ajustando-modelos.html) foi cobrada — o gradiente descendente deixou de ser conveniência e virou a única via até um modelo. É assim também nos capítulos [15](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/index.html) e [16](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/index.html) — mas não em todo o resto: os capítulos [14](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/index.html) e [17](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/index.html) constroem modelos que não descem gradiente nenhum, como o [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/index.html) já avisava.

Há uma segunda herança, e ela só ficou visível na figura desta seção. Os três modelos da escada, mais a máquina de vetores de suporte, decidem todos comparando `dot(x, beta)` com um limiar — e produto escalar contra limiar é, sempre, um **hiperplano no espaço de atributos**. Uma reta, no nosso caso. Os 21 usuários do lado errado dela não estão lá por falta de epochs de treino: estão lá porque nenhuma reta os colocaria do lado certo. O truque do kernel é uma saída para isso, e uma saída cara — ele compra a curvatura trocando o espaço, sem abandonar o produto escalar, e paga com um custo que cresce entre o quadrado e o cubo do número de pontos, e com uma fronteira que ninguém mais consegue ler.

O [Capítulo 14](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/index.html) chega à curvatura por um caminho que não tem nada disso. A árvore de decisão não tem produto escalar, não tem coeficiente, não tem gradiente e não tem ponto de partida aleatório; ela pergunta uma coisa de cada vez — "salário acima de 60.000?" — e a fronteira que sai é feita de degraus paralelos aos eixos. Uma consequência disso fecha justamente a lição da [seção 13.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/03-aplicando-o-modelo.html): para uma árvore, **medir salário em reais ou em desvios padrão é rigorosamente indiferente**. Reescalonar não muda uma única pergunta que ela faça, porque toda pergunta é sobre a **ordem** dos valores de uma coluna, e reescalonar preserva a ordem. O `ValueError` da primeira conta, a saturação em `float64`, a taxa de aprendizado que serve para uma coluna e não para a outra — nada disso existe no próximo capítulo. Some o problema inteiro, junto com o produto escalar que o criava.

> **💡 Dica — Na prática: `scikit-learn`**
>
> O `scikit-learn` traz máquinas de vetores de suporte em `sklearn.svm`, e a escolha entre as classes importa:
>
> ```python
> from sklearn.svm import SVC, LinearSVC
>
> modelo = SVC(kernel='rbf', C=1.0)      # com kernel; fronteira não linear
> modelo = LinearSVC(C=1.0)              # sem kernel; muito mais rápido
> ```
>
> `SVC` é a implementação com kernel, e por baixo dela roda o **LIBSVM**, exatamente o software especializado que a citação acima diz que você deveria usar. O `kernel='rbf'` (*radial basis function*) é o padrão e corresponde a um espaço de dimensão infinita — o caso extremo em que mapear os pontos de verdade seria literalmente impossível e só o truque do kernel torna a conta viável.
>
> Três coisas que valem a pena saber:
>
> - **`C` é o mesmo tipo de hiperparâmetro da regularização** da [seção 12.8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/08-regularizacao.html). Como dados reais quase nunca são separáveis — os nossos não são —, a formulação usada na prática é a de *margem suave*, que permite alguns pontos violarem a margem e cobra por isso. `C` é quanto se cobra.
> - **Reescalonar não é opcional.** A margem é medida com distância euclidiana, então uma coluna em reais e outra em anos produzem o mesmo desastre que a [seção 13.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/03-aplicando-o-modelo.html) mostrou — aqui sem `ValueError`, só com um modelo silenciosamente dominado pela coluna de escala maior. O guia prático do LIBSVM começa por essa recomendação.
> - **`SVC` não escala para dados grandes.** O custo do treino cresce entre o quadrado e o cubo do número de pontos, o que na prática limita `SVC` a algumas dezenas de milhares de exemplos. Acima disso usa-se `LinearSVC` (sem kernel) ou `SGDClassifier`, que aplica a mesma perda de margem por gradiente descendente estocástico — o método do [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/index.html), de volta pela última vez neste capítulo.
>
> E vale registrar a diferença de interface que resume a diferença de critério: `SVC` **não tem** `predict_proba` habilitado por padrão. Ele devolve o lado do hiperplano, não uma probabilidade, porque nunca modelou uma. Obter probabilidades dele exige uma calibração extra por cima, e o `scikit-learn` acabou de mudar como se pede isso: o parâmetro `SVC(probability=True)` foi **depreciado na versão 1.9** e será removido na 1.11, em favor de
>
> ```python
> from sklearn.calibration import CalibratedClassifierCV
>
> modelo = CalibratedClassifierCV(SVC(), ensemble=False)
> ```
>
> A troca melhora o nome, porque agora a classe diz o que faz — antes, um parâmetro booleano escondia um segundo modelo ajustado por baixo. E a ironia sobrevive intacta: essa calibração ajusta **uma regressão logística** sobre as saídas da SVM. O método que este capítulo construiu volta no fim como remendo, para dar à máquina de vetores de suporte a probabilidade que o critério de margem se recusou a modelar.

## Leituras adicionais

A seção "For Further Investigation" do capítulo 16 de Grus (2019) faz duas sugestões.

A primeira é o `scikit-learn`, que tem módulos tanto para [regressão logística](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression) quanto para [máquinas de vetores de suporte](https://scikit-learn.org/stable/modules/svm.html). Os callouts de fechamento de cada seção deste capítulo mostram as duas interfaces.

A segunda é o [LIBSVM](https://www.csie.ntu.edu.tw/~cjlin/libsvm/), a implementação de máquinas de vetores de suporte que o `scikit-learn` usa por baixo em `SVC` e `SVR`. O site traz um guia prático curto — *A Practical Guide to Support Vector Classification* — que vale a leitura justamente por ser prático: ele começa recomendando reescalonar os dados, que é a lição que a [seção 13.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/03-aplicando-o-modelo.html) aprende do jeito difícil.

Para o tratamento estatístico da regressão logística — testes sobre coeficientes, razões de chances, diagnóstico —, James et al. (2021) é o ponto de partida usual, e Hastie et al. (2009) vai fundo tanto na logística quanto no problema de otimização que a máquina de vetores de suporte resolve e que este capítulo não implementa.

## Referências

- **Grus**. *Data Science from Scratch: First Principles with Python*. 2nd ed.. O'Reilly Media. 2019.
- **Hastie; Tibshirani; Friedman**. *The Elements of Statistical Learning*. 2nd ed.. Springer. 2009.
- **James; Witten; Hastie; Tibshirani**. *An Introduction to Statistical Learning*. 2nd ed.. Springer. 2021.